### **Resnet-34 дообученная на Cifar-10**

In [1]:
import torch
import torchvision
from torchvision import transforms
from tempfile import TemporaryDirectory
from torch.utils.data import random_split
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from tqdm import tqdm
import random
from google.colab import files

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


In [3]:
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

g = torch.Generator()
g.manual_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

import os
os.environ['PYTHONHASHSEED'] = str(SEED)

In [4]:
def train_model(model, full_train_loader, full_train_size, criterion, optimizer, num_epochs, device, scheduler):
  best_model_params_path = 'best_model_params.pt'
  torch.save(model.state_dict(), best_model_params_path)
  best_accuracy = 0.0

  for epoch in range(num_epochs):
    print(f'Epoch: {epoch+1}/{num_epochs}')

    for phase in ['train', 'val']:
      if phase=='train':
        model.train()
      else:
        model.eval()

      running_loss = 0.0
      running_corrects = 0

      for inputs, labels in tqdm(full_train_loader[phase]):
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.set_grad_enabled(phase == 'train'):
          outputs = model(inputs)
          _, predictions = torch.max(outputs, 1)
          loss = criterion(outputs, labels)

          if phase=='train':
            loss.backward()
            optimizer.step()


        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(predictions == labels.data)

      epoch_loss = running_loss/full_train_size[phase]
      epoch_accuracy = running_corrects.double()/full_train_size[phase]
      print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_accuracy:.4f}')

      if phase=='val':
        if epoch_accuracy > best_accuracy:
          best_accuracy = epoch_accuracy
          torch.save(model.state_dict(), best_model_params_path)
    scheduler.step()

  print(f'Best val Acc: {best_accuracy:.4f}')
  model.load_state_dict(torch.load(best_model_params_path, weights_only=True))
  return model

In [5]:
def evaluate(model, test_loader, device, criterion):
  model.eval()
  running_loss = 0.0
  correct = 0
  total = 0
  with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Evaluating"):
      inputs, labels = inputs.to(device), labels.to(device)
      outputs = model(inputs)
      loss = criterion(outputs, labels)
      running_loss += loss.item() * inputs.size(0)
      _, predictions = torch.max(outputs, 1)
      total+=labels.size(0)
      correct += predictions.eq(labels).sum().item()
  print()
  print(f'test_loss: {running_loss/len(test_set):.4f}, accuracy: {correct/total:.4f}\n')
  return running_loss/len(test_set), correct/total


In [6]:
transform_train = transforms.Compose([
    transforms.ToTensor(),
    #transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    #transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_train_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform = transform_train)
test_set = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform = transform_test)

train_set, validation_set = random_split(full_train_set, [45000, 5000], generator=g)
full_train_loader = {
    'train': torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True, pin_memory=True, generator=g),
    'val': torch.utils.data.DataLoader(validation_set, batch_size=128, shuffle=False, pin_memory=True),
}
test_loader = torch.utils.data.DataLoader(test_set, batch_size=128, shuffle=False, pin_memory=True)

full_train_size={'train': len(train_set), 'val': len(validation_set)}

100%|██████████| 170M/170M [00:04<00:00, 41.6MB/s]


In [ ]:
model = torchvision.models.resnet34(weights='IMAGENET1K_V1')

for param in model.parameters():
        param.requires_grad = False

model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.fc = nn.Linear(model.fc.in_features, 10)

for param in model.conv1.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            for param in m.parameters():
                param.requires_grad = True

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 89.8MB/s]


In [ ]:
model = model.to(device)
criterion = nn.CrossEntropyLoss()

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.SGD(trainable_params, lr=0.01, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[10, 15], gamma=0.1)

model = train_model(model, full_train_loader, full_train_size, criterion, optimizer, num_epochs=20, device=device, scheduler=scheduler)
loss, accuracy = evaluate(model, test_loader, device, criterion);


Epoch: 1/20


100%|██████████| 352/352 [00:24<00:00, 14.21it/s]


train Loss: 1.4842 Acc: 0.4832


100%|██████████| 40/40 [00:01<00:00, 29.67it/s]


val Loss: 1.3516 Acc: 0.5414
Epoch: 2/20


...

train Loss: 0.3398 Acc: 0.8799


100%|██████████| 40/40 [00:01<00:00, 30.55it/s]


val Loss: 0.5546 Acc: 0.8124
Best val Acc: 0.8140


Evaluating: 100%|██████████| 79/79 [00:02<00:00, 29.87it/s]


test_loss: 0.5623, accuracy: 0.8113



In [ ]:
import datetime
file = open('log.txt', "a", encoding = "utf-8")
dt = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
file.write(f"[{dt}] SEED: {SEED} Accuracy of model: {accuracy:.4f}\n");
file.close()

torch.save(model.state_dict(), "Model.pth")
files.download("Model.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### **PGD атака**

In [7]:
def PGD_attack(model, image, label, epsilon, alpha, num_steps, device):
    label = label.to(device)
    image = image.to(device)

    original_image = image.clone().detach()

    noise = torch.empty_like(original_image).uniform_(-epsilon, epsilon)
    perturbed_image = original_image + noise
    perturbed_image = torch.clamp(perturbed_image, 0, 1).detach()

    for _ in range(num_steps):
        perturbed_image.requires_grad = True

        output = model(perturbed_image)
        loss = nn.functional.cross_entropy(output, label)

        model.zero_grad()
        loss.backward()

        with torch.no_grad():
            sign_data_grad = perturbed_image.grad.sign()
            perturbed_image = perturbed_image + alpha * sign_data_grad
            perturbed_image = torch.max(torch.min(perturbed_image, original_image + epsilon), original_image - epsilon)
            perturbed_image = torch.clamp(perturbed_image, 0, 1)

        perturbed_image = perturbed_image.detach()

    return perturbed_image.detach()

In [8]:
def PGD_test(model, device, test_loader, epsilon, alpha, num_steps):
  robust_correct = 0
  total = 0
  model.eval()

  for data, target in tqdm(test_loader, desc=f'(eps={epsilon})'):
    data, target = data.to(device), target.to(device)

    perturbed_data = PGD_attack(model, data, target, epsilon, alpha, num_steps, device)

    with torch.no_grad():
        output_adv = model(perturbed_data)
        pred = output_adv.argmax(dim=1)
        robust_correct += (pred == target).sum().item()

    total += data.size(0)

  robust_acc = robust_correct / total
  print(f"\nEps: {epsilon} | Robust Accuracy: {robust_acc:.4f}\n")

  return robust_acc

In [ ]:
EPS = 8/255
ALPH = 1/255
NUM_STEPS = 20

acc = PGD_test(model, device, test_loader, EPS, ALPH, NUM_STEPS)

(eps=0.03137254901960784): 100%|██████████| 79/79 [01:28<00:00,  1.12s/it]


Eps: 0.03137254901960784 | Robust Accuracy: 0.0000



### **PGD защита**

In [9]:
def adversarial_train_model(model, full_train_loader, criterion, optimizer, scheduler, num_epochs, adversarial=False, eps=0.03, alpha=0.01, pgd_iters=10):
  best_model_params_path = 'best_model_params.pt'
  torch.save(model.state_dict(), best_model_params_path)
  best_accuracy = 0.0

  for epoch in range(num_epochs):
    print(f'Epoch: {epoch+1}/{num_epochs}')

    for phase in ['train', 'val']:
      if phase == 'train':
        model.train()
      else:
        model.eval()

      running_loss = 0.0
      running_corrects = 0

      for inputs, labels in tqdm(full_train_loader[phase]):
        inputs = inputs.to(device)
        labels = labels.to(device)

        if adversarial and phase == 'train':
          model.eval()
          with torch.enable_grad():
            adv_inputs = PGD_attack(model, inputs, labels, eps, alpha, pgd_iters, device)
          model.train()
          inputs = adv_inputs

        optimizer.zero_grad()

        with torch.set_grad_enabled(phase == 'train'):
          outputs = model(inputs)
          _, predictions = torch.max(outputs, 1)
          loss = criterion(outputs, labels)

          if phase == 'train':
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(predictions == labels.data)

      epoch_loss = running_loss / full_train_size[phase]
      epoch_accuracy = running_corrects.double() / full_train_size[phase]
      print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_accuracy:.4f}')

      if phase == 'val':

        if epoch_accuracy > best_accuracy:
          best_accuracy = epoch_accuracy
          torch.save(model.state_dict(), best_model_params_path)
    scheduler.step()

  print(f'Best val Acc: {best_accuracy:.4f}\n')
  model.load_state_dict(torch.load(best_model_params_path, weights_only=True))
  return model

### **Применение PGD защиты**

In [ ]:
model = torchvision.models.resnet34(weights='IMAGENET1K_V1')

for param in model.parameters():
        param.requires_grad = False

model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.fc = nn.Linear(model.fc.in_features, 10)

for param in model.conv1.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            for param in m.parameters():
                param.requires_grad = True

In [ ]:
EPS = 8/255
ALPH = 1/255
NUM_STEPS = 20

model = model.to(device)
criterion = nn.CrossEntropyLoss()

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.SGD(trainable_params, lr=0.01, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[10, 15], gamma=0.1)

model = adversarial_train_model(model, full_train_loader, criterion, optimizer, scheduler, num_epochs=20, adversarial=True, eps=8/255, alpha=1/255, pgd_iters= 20)
loss, usual_acc = evaluate(model, test_loader, device, criterion);
robust_acc = PGD_test(model, device, test_loader, EPS, ALPH, NUM_STEPS)
print(f"PGD degradation: {accuracy-usual_acc:.4f}\n")

Epoch: 1/20


100%|██████████| 352/352 [06:46<00:00,  1.15s/it]


train Loss: 2.2497 Acc: 0.1816


100%|██████████| 40/40 [00:01<00:00, 25.06it/s]


val Loss: 1.8653 Acc: 0.3246
Epoch: 2/20


...

val Loss: 1.5017 Acc: 0.4830
Best val Acc: 0.4836



Evaluating: 100%|██████████| 79/79 [00:03<00:00, 25.20it/s]



test_loss: 1.4992, accuracy: 0.4825



(eps=0.03137254901960784): 100%|██████████| 79/79 [01:27<00:00,  1.11s/it]


Eps: 0.03137254901960784 | Robust Accuracy: 0.2937

PGD degradation: 0.3288



In [ ]:
dt = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
file = open('log.txt', "a", encoding = "utf-8")
file.write(f"[{dt}] (PGD-based Adversarial Training) SEED: {SEED} Accuracy on usual images: {usual_acc:.4f} Accuracy degradation: {accuracy-usual_acc:.4f}\n");
file.close()

torch.save(model.state_dict(), "PGD.pth")
files.download("PGD.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### **Label Smoothing защита**

In [11]:
def smooth_labels(labels, smoothing_param, num_classes):
    smooth_labels = labels * (1 - smoothing_param) + smoothing_param / num_classes
    return smooth_labels

In [12]:
def label_smoothing_train(model, full_train_loader, full_train_size, criterion, optimizer, scheduler, num_epochs, smoothing_param, device):
  best_model_params_path = 'best_model_params.pt'
  torch.save(model.state_dict(), best_model_params_path)
  best_accuracy = 0.0

  for epoch in range(num_epochs):
    print(f'Epoch: {epoch+1}/{num_epochs}')

    for phase in ['train', 'val']:
      if phase=='train':
        model.train()
      else:
        model.eval()

      running_loss = 0.0
      running_corrects = 0

      for inputs, labels in tqdm(full_train_loader[phase]):
        inputs = inputs.to(device)
        labels = labels.to(device)

        labels_one_hot = nn.functional.one_hot(labels, num_classes=10).float()
        smoothed_labels = smooth_labels(labels_one_hot, smoothing_param, 10)

        optimizer.zero_grad()

        with torch.set_grad_enabled(phase == 'train'):
          outputs = model(inputs)
          _, predictions = torch.max(outputs, 1)
          loss = criterion(outputs, smoothed_labels)

          if phase=='train':
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(predictions == labels.data)

      epoch_loss = running_loss/full_train_size[phase]
      epoch_accuracy = running_corrects.double()/full_train_size[phase]
      print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_accuracy:.4f}')

      if phase=='val':
        if epoch_accuracy > best_accuracy:
          best_accuracy = epoch_accuracy
          torch.save(model.state_dict(), best_model_params_path)
      print(f"Current LR: {optimizer.param_groups[0]['lr']:.2e}")

  print(f'Best val Acc: {best_accuracy:.4f}')
  model.load_state_dict(torch.load(best_model_params_path, weights_only=True))
  return model

### **Применение Label Smoothing защиты**

In [ ]:
model = torchvision.models.resnet34(weights='IMAGENET1K_V1')

for param in model.parameters():
        param.requires_grad = False

model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.fc = nn.Linear(model.fc.in_features, 10)

for param in model.conv1.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            for param in m.parameters():
                param.requires_grad = True

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 238MB/s]


In [ ]:
model = model.to(device)
criterion = nn.CrossEntropyLoss()

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.SGD(trainable_params, lr=0.01, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[10, 15], gamma=0.1)

model = label_smoothing_train(model, full_train_loader, full_train_size, criterion, optimizer, scheduler, num_epochs=20, smoothing_param=0.5, device=device)
loss, acc = evaluate(model, test_loader, device, criterion);
print(f"Accuracy degradation: {accuracy-acc:.4f}\n")

Epoch: 1/20


100%|██████████| 352/352 [00:25<00:00, 13.87it/s]


train Loss: 2.1827 Acc: 0.4033
Current LR: 1.00e-02


100%|██████████| 40/40 [00:01<00:00, 29.03it/s]


val Loss: 2.1336 Acc: 0.4714
Current LR: 1.00e-02
Epoch: 2/20


...

train Loss: 1.8493 Acc: 0.8202
Current LR: 1.00e-02


100%|██████████| 40/40 [00:01<00:00, 28.96it/s]


val Loss: 1.8775 Acc: 0.7750
Current LR: 1.00e-02
Best val Acc: 0.7860


Evaluating: 100%|██████████| 79/79 [00:03<00:00, 24.87it/s]


test_loss: 1.0883, accuracy: 0.7827

Accuracy degradation: 0.0286



In [ ]:
dt = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
file = open('log.txt', "a", encoding = "utf-8")
file.write(f"[{dt}] SEED: {SEED} LS accuracy: {acc:.4f} Accuracy degradation: {accuracy - acc:.4f}\n");
file.close()

torch.save(model.state_dict(), "LS.pth")
files.download("LS.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### **Комбинация состязательного обучения и Label Smoothing**

In [13]:
def ls_and_pgd_train(model, full_train_loader, full_train_size, criterion, optimizer, scheduler, num_epochs, adversarial=False, eps=0.03, alpha=0.01, pgd_iters=10, smoothing_param=0.5):
  best_model_params_path = 'best_model_params.pt'
  torch.save(model.state_dict(), best_model_params_path)
  best_accuracy = 0.0

  for epoch in range(num_epochs):
    print(f'Epoch: {epoch+1}/{num_epochs}')

    for phase in ['train', 'val']:
      if phase == 'train':
        model.train()
      else:
        model.eval()

      running_loss = 0.0
      running_corrects = 0

      for inputs, labels in tqdm(full_train_loader[phase]):
        inputs = inputs.to(device)
        labels = labels.to(device)

        labels_one_hot = nn.functional.one_hot(labels, num_classes=10).float()
        smoothed_labels = smooth_labels(labels_one_hot, smoothing_param, 10)

        if adversarial and phase == 'train':
          model.eval()
          with torch.enable_grad():
            adv_inputs = PGD_attack(model, inputs, labels, eps, alpha, pgd_iters, device)
          model.train()
          inputs = adv_inputs

        optimizer.zero_grad()

        with torch.set_grad_enabled(phase == 'train'):
          outputs = model(inputs)
          _, predictions = torch.max(outputs, 1)
          loss = criterion(outputs, smoothed_labels)

          if phase == 'train':
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(predictions == labels.data)

      epoch_loss = running_loss / full_train_size[phase]
      epoch_accuracy = running_corrects.double() / full_train_size[phase]
      print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_accuracy:.4f}')

      if phase == 'val':

        if epoch_accuracy > best_accuracy:
          best_accuracy = epoch_accuracy
          torch.save(model.state_dict(), best_model_params_path)
    scheduler.step()

  print(f'Best val Acc: {best_accuracy:.4f}')
  model.load_state_dict(torch.load(best_model_params_path, weights_only=True))
  return model

### **Применение**

In [14]:
model = torchvision.models.resnet34(weights='IMAGENET1K_V1')

for param in model.parameters():
        param.requires_grad = False

model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.fc = nn.Linear(model.fc.in_features, 10)

for param in model.conv1.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            for param in m.parameters():
                param.requires_grad = True

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 188MB/s]


In [15]:
model = model.to(device)
criterion = nn.CrossEntropyLoss()

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.SGD(trainable_params, lr=0.01, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[10, 15], gamma=0.1)

model = ls_and_pgd_train(model, full_train_loader, full_train_size, criterion, optimizer, scheduler, num_epochs=20, adversarial=True, eps=8/255, alpha=1/255, pgd_iters= 20, smoothing_param=0.5)
loss, acc = evaluate(model, test_loader, device, criterion);
print(f"Accuracy degradation: {accuracy-acc:.4f}\n")

Epoch: 1/20


100%|██████████| 352/352 [06:41<00:00,  1.14s/it]


train Loss: 2.3779 Acc: 0.0969


100%|██████████| 40/40 [00:01<00:00, 28.11it/s]


val Loss: 2.2531 Acc: 0.2590
Epoch: 2/20


...

train Loss: 2.2253 Acc: 0.2680


100%|██████████| 40/40 [00:01<00:00, 29.36it/s]


val Loss: 2.1549 Acc: 0.3954
Best val Acc: 0.3970


Evaluating: 100%|██████████| 79/79 [00:02<00:00, 29.32it/s]


test_loss: 1.9082, accuracy: 0.4131

Accuracy degradation: 0.3982



In [16]:
dt = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
file = open('log.txt', "a", encoding = "utf-8")
file.write(f"[{dt}] SEED: {SEED} PGD and LS combination accuracy: {acc:.4f} Accuracy degradation: {accuracy-acc:.4f}\n");
file.close()

torch.save(model.state_dict(), "LS_and_PGD.pth")
files.download("LS_and_PGD.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### **Закрытие и скачивание файла логирования**

In [17]:
files.download('log.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>